In [11]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
sup = pd.read_csv('aaus6_class_suppliers - aaus6_class_suppliers.csv')
con = pd.read_csv('aaus6_class_contracts - aaus6_class_contracts.csv')
po = pd.read_csv('aaus6_class_purchase_orders - aaus6_class_purchase_orders.csv')
rec = pd.read_csv('aaus6_class_goods_receipts - aaus6_class_goods_receipts.csv')
print([len(sup), len(con), len(po), len(rec)])

[60, 96, 288, 354]


In [17]:
print("suppliers: ", sup.columns)
print("contracts: ", con.columns)
print("purchase orders: ", po.columns)
print("goods receipts: ", rec.columns)
print('Unique supplier keys:', sup.supplier_key.nunique())
print('Unique contract keys:', con.contract_key.nunique())
print('Unique purchase order keys:', po.order_key.nunique())
print('Unique goods receipt keys:', rec.receipt_key.nunique())


suppliers:  Index(['supplier_key', 'supplier_name', 'supplier_region', 'certification',
       'service_focus', 'rating'],
      dtype='object')
contracts:  Index(['contract_key', 'supplier_key', 'procurement_method',
       'contract_category', 'contract_amount_etb', 'approval_stage',
       'start_date'],
      dtype='object')
purchase orders:  Index(['order_key', 'contract_key', 'purchase_unit', 'item_category',
       'order_date', 'order_amount_etb', 'priority', 'approval_status'],
      dtype='object')
goods receipts:  Index(['receipt_key', 'order_key', 'received_date', 'quantity_accepted',
       'inspection_result', 'receiving_office'],
      dtype='object')
Unique supplier keys: 60
Unique contract keys: 96
Unique purchase order keys: 288
Unique goods receipt keys: 354


In [24]:
print('Purchase order unique contract kyes:', po.contract_key.nunique())

Purchase order unique contract kyes: 102


In [25]:
# Execute inner join on contract_key
joined_inner = po.merge(
  con,
on='contract_key',
how='inner'
)
print('Orders:', len(po))
print('Inner result:', len(joined_inner))
print('Lost orders:', 
len(po)-len(joined_inner))

Orders: 288
Inner result: 279
Lost orders: 9


In [31]:
# Execute left join on contract_key
joined_left = po.merge(
  con,
on='contract_key',
how='left'
)
print('Orders:', len(po))
print('Inner result:', len(joined_left))
print('Lost orders:', 
len(po)-len(joined_left))

joined_left.shape

Orders: 288
Inner result: 288
Lost orders: 0


(288, 14)

In [32]:
# Execute right join on contract_key
joined_right = po.merge(
  con,
on='contract_key',
how='right'
)
print('Orders:', len(po))
print('Inner result:', len(joined_right))
print('Lost orders:', 
len(po)-len(joined_right))

joined_right.shape

Orders: 288
Inner result: 282
Lost orders: 6


(282, 14)

In [34]:
right_view = po.merge(
con, on='contract_key',
how='right'
)
print('Right rows:', len(right_view))
print('Contract IDs:',
right_view.contract_key.nunique())
print('Orders missing:',
right_view.order_key.isna().sum())
right_view.shape

Right rows: 282
Contract IDs: 96
Orders missing: 3


(282, 14)

In [38]:
outer = po.merge(
con, on='contract_key',
how='outer', indicator=True
)
print(outer['_merge']
.value_counts().to_dict())
print('Rows:', len(outer))

outer

{'both': 279, 'left_only': 9, 'right_only': 3}
Rows: 291


,order_key,contract_key,purchase_unit,item_category,order_date,order_amount_etb,priority,approval_status,supplier_key,procurement_method,contract_category,contract_amount_etb,approval_stage,start_date,_merge
0,PO-00035,C-0001,Lideta,Solar Controller,2026-07-26,61635.0,Urgent,Released,S-007,RFQ,Water Kiosk,139317.0,Active,2026-01-02,both
1,PO-00131,C-0001,Lideta,Solar Controller,2026-09-04,42691.0,Urgent,Released,S-007,RFQ,Water Kiosk,139317.0,Active,2026-01-02,both
2,PO-00227,C-0001,Lideta,Solar Controller,2026-03-08,118747.0,Urgent,Released,S-007,RFQ,Water Kiosk,139317.0,Active,2026-01-02,both
3,PO-00070,C-0002,Gullele,Solar Controller,2026-06-11,98270.0,High,Pending,S-014,Framework,SCADA Gateway,158634.0,Under Review,2026-01-03,both
4,PO-00166,C-0002,Gullele,Solar Controller,2026-07-21,79326.0,High,Pending,S-014,Framework,SCADA Gateway,158634.0,Under Review,2026-01-03,both
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
286,PO-00160,C-X160,Gullele,SCADA Gateway,2026-06-21,56760.0,High,Pending,NaN,NaN,NaN,NaN,NaN,NaN,left_only
287,PO-00192,C-X192,Merkato,Water Kiosk,2026-04-22,82112.0,Routine,Approved,NaN,NaN,NaN,NaN,NaN,NaN,left_only
288,PO-00224,C-X224,Akaki,Lab Analyzer,2026-02-21,107464.0,Urgent,Released,NaN,NaN,NaN,NaN,NaN,NaN,left_only
289,PO-00256,C-X256,Gullele,SCADA Gateway,2026-07-31,37816.0,High,Pending,NaN,NaN,NaN,NaN,NaN,NaN,left_only


In [39]:
audit = po.merge(
con, on='contract_key',
how='left', indicator=True,
validate='many_to_one'
)
print(audit['_merge']
.value_counts().to_dict())
print('Unmatched keys:', audit.loc[
audit['_merge']=='left_only',
'contract_key'].nunique())

{'both': 279, 'left_only': 9, 'right_only': 0}
Unmatched keys: 9


In [40]:
miss = audit.loc[audit['_merge']=='left_only']
print('Rows needing review:', len(miss))
print('Units:', miss.purchase_unit.value_counts().
head(3).to_dict())

Rows needing review: 9
Units: {'Akaki': 3, 'Gullele': 3, 'Merkato': 3}


In [42]:
semi = po.loc[po.contract_key.isin(
con.contract_key)].copy()
print('Matched orders:', len(semi))
print('Unmatched removed:', 
len(po)-len(semi))

Matched orders: 279
Unmatched removed: 9


In [43]:
anti = po.loc[~po.contract_key.isin(
con.contract_key)].copy()
print('Review rows:', len(anti))
print('Reconciles:', 
len(semi)+len(anti)==len(po))

Review rows: 9
Reconciles: True


In [45]:
anti = po.loc[~po.contract_key.isin(
con.contract_key)].copy()
print('Review rows:', len(anti))
print('Reconciles:', 
len(semi)+len(anti)==len(po))

Review rows: 9
Reconciles: True


In [46]:
legacy = sup.rename(columns={
'supplier_key':'vendor_reference'})
map_view = con.merge(legacy,
left_on='supplier_key',
right_on='vendor_reference',
how='left', indicator=True)
print(map_view['_merge'].value_counts().to_dict())

{'both': 92, 'left_only': 4, 'right_only': 0}


In [49]:
left_temp = po.rename(columns={'priority':'status'})
right_temp = con.rename(columns={'approval_stage':'status'})
sfx = left_temp.merge(right_temp, 
on='contract_key',
how='left', suffixes=('_order','_contract'))
print([x for x in sfx if x.startswith('status')])
po

['status_order', 'status_contract']


,order_key,contract_key,purchase_unit,item_category,order_date,order_amount_etb,priority,approval_status
0,PO-00001,C-0011,Bole,Water Kiosk,2026-02-06,28761,High,Pending
1,PO-00002,C-0022,Akaki,Water Kiosk,2026-02-11,32522,Urgent,Released
2,PO-00003,C-0033,Kirkos,SCADA Gateway,2026-02-16,36283,Routine,Approved
3,PO-00004,C-0044,Gullele,SCADA Gateway,2026-02-21,40044,High,Pending
4,PO-00005,C-0055,Lideta,SCADA Gateway,2026-02-26,43805,Urgent,Released
...,...,...,...,...,...,...,...,...
283,PO-00284,C-0052,Akaki,Lab Analyzer,2026-05-12,48124,Urgent,Released
284,PO-00285,C-0063,Kirkos,Solar Controller,2026-05-17,51885,Routine,Approved
285,PO-00286,C-0074,Gullele,Solar Controller,2026-05-22,55646,High,Pending
286,PO-00287,C-0085,Lideta,Solar Controller,2026-05-27,59407,Urgent,Released


In [52]:
enriched = (po.merge(con, on='contract_key',
how='left', validate='many_to_one')
.merge(sup, on='supplier_key', how='left',
validate='many_to_one'))
print('Rows:', len(enriched))
print('Known regions:', 
enriched.supplier_region.notna().sum())
enriched.shape

Rows: 288
Known regions: 270


(288, 19)

In [53]:
kiosk = (enriched.loc[
    enriched.item_category == 'Water Kiosk'
].copy())
print('Kiosk-related orders:', len(kiosk))
print('Known suppliers:',
    kiosk.supplier_name.notna().sum())

Kiosk-related orders: 72
Known suppliers: 60


In [55]:
stage1 = po.merge(con, on='contract_key',how='left')
stage2 = stage1.merge(sup, on='supplier_key',how='left', indicator=True)
print(stage2['_merge'].value_counts().to_dict())

{'both': 270, 'left_only': 18, 'right_only': 0}


In [56]:
po_rec = po.merge(rec, on='order_key',
    how='left', validate='one_to_many')
print('Purchase orders:', len(po))
print('Rows after join:', len(po_rec))
print('Orders without receipt:',
      po_rec.receipt_key.isna().sum())

Purchase orders: 288
Rows after join: 336
Orders without receipt: 0


In [57]:
gr_summary = rec.groupby('order_key', 
as_index=False).agg(
accepted_qty=('quantity_accepted', 'sum'),
receipt_count=('receipt_key', 'count'))
order_summary = po.merge(gr_summary, on='order_key',
how='left', validate='one_to_one')
print(len(order_summary), 
order_summary.receipt_count.notna().sum())

288 288
